In [28]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

## Helper Functions

In [82]:
class BasisFunctions:
	def basis_function_1d(descriptor, a, b, c, d):
		return (
			a + 
			b*descriptor[0] + 
			c*descriptor[1] + 
			d*descriptor[0]*descriptor[1]
		)

	def basis_function_2d(descriptor, a, b, c, d, e, f):
		return (
			a + 
			b*descriptor[0] + 
			c*descriptor[1] + 
			d*descriptor[0]*descriptor[1] + 
			e*descriptor[0]**2 + 
			f*descriptor[1]**2
		)

	def basis_function_2d_x2y(descriptor, a, b, c, d, e, f, g, h, i):
		return (
			a + 
			b*descriptor[0] + 
			c*descriptor[1] + 
			d*descriptor[0]*descriptor[1] + 
			e*descriptor[0]**2 + 
			f*descriptor[1]**2 +
			g*descriptor[0]**2*descriptor[1] + 
			h*descriptor[1]**2*descriptor[0] +
			i*descriptor[1]**2*descriptor[0]**2
		)

	def basis_function_3d(descriptor, a, b, c, d, e, f, g, h):
		return (
			a + 
			b*descriptor[0] + 
			c*descriptor[1] + 
			d*descriptor[0]*descriptor[1] + 
			e*descriptor[0]**2 + 
			f*descriptor[1]**2 +
			g*descriptor[0]**3 +
			h*descriptor[1]**3
		)

	def basis_function_3d_sin(descriptor, a, b, c, d, e, f, g, h, i, j):
		return (
			a + 
			b*descriptor[0] + 
			c*descriptor[1] + 
			d*descriptor[0]*descriptor[1] + 
			e*descriptor[0]**2 + 
			f*descriptor[1]**2 +
			g*descriptor[0]**3 +
			h*descriptor[1]**3 +
			i*np.sin(descriptor[0]) +
			j*np.sin(descriptor[1]) 
		)

	def basis_function_3d_x2y(descriptor, a, b, c, d, e, f, g, h, i, j, k):
		return (
			a + 
			b*descriptor[0] + 
			c*descriptor[1] + 
			d*descriptor[0]*descriptor[1] + 
			e*descriptor[0]**2 + 
			f*descriptor[1]**2 +
			g*descriptor[0]**3 +
			h*descriptor[1]**3 +
			i*descriptor[0]**2*descriptor[1] +
			j*descriptor[1]**2*descriptor[0] +
			k*descriptor[1]**2*descriptor[0]**2
		)

	def basis_function_4d(descriptor, a, b, c, d, e, f, g, h, i, j):
		return (
			a + 
			b*descriptor[0] + 
			c*descriptor[1] + 
			d*descriptor[0]*descriptor[1] + 
			e*descriptor[0]**2 + 
			f*descriptor[1]**2 +
			g*descriptor[0]**3 +
			h*descriptor[1]**3 +
			i*descriptor[0]**4 +
			j*descriptor[1]**4
		)

	def basis_function_exp(descriptor, a, b, c, d, e):
		return (
			a + 
			b*np.exp(descriptor[0]) + 
			c*np.exp(descriptor[1]) + 
			d*(1/np.exp(descriptor[0])) + 
			e*(1/np.exp(descriptor[1]))
		)

	def basis_function_log(descriptor, a, b, c, d, e):
		return (
			a + 
			b*np.log(descriptor[0]) + 
			c*np.log(descriptor[1]) + 
			d*(1/np.log(descriptor[0])) + 
			e*(1/np.log(descriptor[1]))
		)

	functions = {
		'1d': basis_function_1d,
		'2d': basis_function_2d,
		'2d_x2y': basis_function_2d_x2y,
		'3d': basis_function_3d,
		'3d_sin': basis_function_3d_sin,
		'3d_x2y': basis_function_3d_x2y,
		'4d': basis_function_4d,
		'exp': basis_function_exp,
		'log': basis_function_log
	}

	def _calculate_rmse(self, descriptor, target, basis_function, params):
		pred = basis_function(descriptor.T, *params)
		return root_mean_squared_error(target, pred)

	def compare(self, descriptor, target, n_splits=5):
		kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
		results = {name: [] for name in self.functions.keys()}
		
		for train_index, test_index in kf.split(descriptor):
			X_train, X_test = descriptor[train_index], descriptor[test_index]
			y_train, y_test = target[train_index], target[test_index]
			
			for name,func in self.functions.items():
				params, _ = curve_fit(func, X_train.T, y_train[:, 0])
				mse = self._calculate_rmse(X_test, y_test, func, params)
				results[name].append(mse)

		print("5 K-Fold")
		
		for func_name, mse_list in results.items():
			print(f"{func_name}:")
			print(f"  Mean MSE = {np.mean(mse_list)}")
			print(f"  Std MSE = {np.std(mse_list)}")
		
		return results

	def fit(self, descriptor, target, basis_function):		
		params, _ = curve_fit(basis_function, descriptor.T, target[:, 0])
	
		mse = self._calculate_rmse(descriptor, target, basis_function, params)
		print(f"{basis_function.__name__}: RMSE = {mse}")
	
		return params
	
	def predict(self, coords, params, basis_function, size_x, size_y):		
		return basis_function(coords.T, *params).reshape(size_x, size_y)

In [83]:
class Plotter:
	def scatter_plot(self, x, y, z, color='blue'):
		return go.Scatter3d(
			x=x, 
			y=y, 
			z=z, 
			mode='markers', 
  			marker=dict(size=2, color=color),
		)
	
	def surface_plot(self, x, y, z, color='blue'):
		return go.Surface(
			x=x, 
			y=y, 
			z=z, 
			opacity=0.5,
    		colorscale=[color, color], 
    		showscale=False,
		)

	def plot(self, data, title):
		fig = go.Figure(data=data)
		fig.update_layout(width=800,height=600)
		fig.update_layout(scene_aspectmode='cube')
		fig.update_layout(
			scene=dict(
				xaxis=dict(title='al (wt%)'),
				yaxis=dict(title='si (wt%)'),
				zaxis=dict(title='K1 (10^3 erg/cm^3)')
			)
		)
		fig.update_layout(
			title={
				'text': title,
				'x': 0.5,
				'y': 0.9,
				'xanchor': 'center',
				'yanchor': 'top',
				'font': dict(size=30)
			}
		)
		fig.update_layout(legend=dict(font=dict(size=20)))

		fig.update_layout(
			legend=go.layout.Legend(
				itemsizing='constant'
			)
		)

		fig.show()

In [84]:
class AnalyzeTernaryData:
	def __init__(
			self, 
			raw_data_path, 
			basis_functions: BasisFunctions,
			plotter: Plotter,
			al_range = [4, 7], 
			si_range = [7, 12],
			al_diff = 100,
			si_diff = 100,
		):
		self.raw_data_path = raw_data_path
		self.basis_functions = basis_functions
		self.plotter = plotter
		self.al_range = al_range
		self.si_range = si_range
		self.al_diff = al_diff
		self.si_diff = si_diff

		self._load_raw_data()
		self._initialize_descriptors_and_targets()
		self._initialize_coordinates()

	def _load_raw_data(self):
		self.raw_data = pd.read_csv(self.raw_data_path)

	def _initialize_descriptors_and_targets(self):
		self.descriptors = self.raw_data[['al', 'si']].values
		self.targets = self.raw_data[['k1']].values

	def _initialize_coordinates(self):
		self.al_1d = np.linspace(self.al_range[0], self.al_range[1], self.al_diff)
		self.si_1d = np.linspace(self.si_range[0], self.si_range[1], self.si_diff)

		self.al, self.si = np.meshgrid(self.al_1d, self.si_1d)
		self.coords = np.column_stack((self.al.ravel(), self.si.ravel()))

	def ols_compare_basis_functions(self):
		self.basis_functions.compare(self.descriptors, self.targets)

	def _ols_plot_basis_functions(self, basis_function, name):
		params = self.basis_functions.fit(self.descriptors, self.targets, basis_function)

		pred = self.basis_functions.predict(self.coords, params, basis_function, self.al_diff, self.si_diff)

		scatter = self.plotter.scatter_plot(
					self.descriptors[:,0], 
					self.descriptors[:,1], 
					self.targets[:,0]
				)
		
		surface = self.plotter.surface_plot(
					self.al_1d, 
					self.si_1d, 
					pred
				)

		self.plotter.plot([scatter, surface], name)

	def ols_plot_basis_functions(self):
		for name, basis_function in self.basis_functions.functions.items():
			self._ols_plot_basis_functions(basis_function, name)

In [85]:
plotter = Plotter()
basis_functions = BasisFunctions()

## Analysis

### 900

In [86]:
analyzer_900 = AnalyzeTernaryData("Sendust Data(900).csv", basis_functions, plotter)

In [87]:
analyzer_900.ols_compare_basis_functions()

5 K-Fold
1d:
  Mean MSE = 3.0790157736600703
  Std MSE = 0.21795331261999987
2d:
  Mean MSE = 2.7137842350620596
  Std MSE = 0.12496859838827987
2d_x2y:
  Mean MSE = 2.723840491672369
  Std MSE = 0.21109150953484984
3d:
  Mean MSE = 1.9385971288116841
  Std MSE = 0.1736765015766257
3d_sin:
  Mean MSE = 1.6741131236312659
  Std MSE = 0.26012839986302694
3d_x2y:
  Mean MSE = 1.9187662941849433
  Std MSE = 0.1855508034740262
4d:
  Mean MSE = 1.6428311091492538
  Std MSE = 0.19340574468546617
exp:
  Mean MSE = 4.190224993518105
  Std MSE = 0.1846810807767863
log:
  Mean MSE = 3.9104663187099478
  Std MSE = 0.1759593557069958


In [88]:
analyzer_900.ols_plot_basis_functions()

basis_function_1d: RMSE = 3.041797618217917


basis_function_2d: RMSE = 2.670322961309778


basis_function_2d_x2y: RMSE = 2.604317094941036


basis_function_3d: RMSE = 1.8734548243064975


basis_function_3d_sin: RMSE = 1.6078167832739825


basis_function_3d_x2y: RMSE = 1.8248150836132724


basis_function_4d: RMSE = 1.5813942673901618


basis_function_exp: RMSE = 4.100234655117487


basis_function_log: RMSE = 3.8192097555610647


## 1200

In [90]:
analyzer_1200 = AnalyzeTernaryData("Sendust Data(1200).csv", basis_functions, plotter)

In [91]:
analyzer_1200.ols_compare_basis_functions()

5 K-Fold
1d:
  Mean MSE = 1.8054102440071726
  Std MSE = 0.2837347682145764
2d:
  Mean MSE = 1.2183699721905723
  Std MSE = 0.16509101375197438
2d_x2y:
  Mean MSE = 0.9291930303983197
  Std MSE = 0.13501912547945452
3d:
  Mean MSE = 1.1006432831602075
  Std MSE = 0.08333132589102887
3d_sin:
  Mean MSE = 1.0982504138594682
  Std MSE = 0.0791027037459755
3d_x2y:
  Mean MSE = 0.870900947833517
  Std MSE = 0.11932833943652815
4d:
  Mean MSE = 1.0874033211274594
  Std MSE = 0.08456633994353356
exp:
  Mean MSE = 2.4918107991981304
  Std MSE = 0.29237580803183955
log:
  Mean MSE = 2.4222266924469116
  Std MSE = 0.297641311809532


In [92]:
analyzer_1200.ols_plot_basis_functions()

basis_function_1d: RMSE = 1.7841642650863445


basis_function_2d: RMSE = 1.178991484294164


basis_function_2d_x2y: RMSE = 0.8864867604252189


basis_function_3d: RMSE = 1.0304725988021521


basis_function_3d_sin: RMSE = 1.013261681782008


basis_function_3d_x2y: RMSE = 0.7885831654225793


basis_function_4d: RMSE = 1.0070450540019376


basis_function_exp: RMSE = 2.393423696813608


basis_function_log: RMSE = 2.3432625416335635
